# 03 Inference Demo

This notebook reads the trained Transformer checkpoint and learned PSL rule weights produced by `02_training_demo.ipynb`. It does not start a new training run.

In [ ]:
from collections import defaultdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.transformer import load_checkpoint
from utils.config import DatasetConfig, inference_dir, result_dir as build_result_dir
from utils.io import load_json, load_psl

config = DatasetConfig(name="mnist-1", train_size=1000, valid_size=1000, inference_size=5000, overlap=0.0)
data_dir = inference_dir(config)
result_dir = build_result_dir(config)

model_path = result_dir / "model.pt"
runtime_output = result_dir / "runtime-output.json"
learned_rules_path = result_dir / "learned-rules.json"

required = [model_path, runtime_output, learned_rules_path]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks/02_training_demo.ipynb first. Missing: " + ", ".join(str(path) for path in missing))

model_path, runtime_output, learned_rules_path

## Load Trained Artifacts

The Transformer checkpoint stores the neural model after PSL-gradient training. `learned-rules.json` stores the final weighted PSL rules returned by the runtime.

In [ ]:
model = load_checkpoint(model_path)
payload = load_json(runtime_output)
learned_rules = load_json(learned_rules_path)

print("evaluations:", payload.get("evaluations", []))
print("atom count:", len(payload.get("atoms", [])))
print("learned rule count:", len(learned_rules))
print("\nLearned rule excerpt:")
for rule in learned_rules[:5]:
    print("-", rule)

## Parse Runtime Atoms

`NeuralClassifier(image, digit)` gives Transformer/DeepPredicate digit probabilities. `ImageSum(image_1, image_2, sum)` gives the PSL-fused addition soft label.

In [ ]:
def collect_atoms(payload):
    neural = defaultdict(dict)
    image_sum = defaultdict(dict)
    for atom in payload.get("atoms", []):
        predicate = atom["predicate"]
        args = [int(value) for value in atom["arguments"]]
        if predicate == "NEURALCLASSIFIER":
            image_id, digit = args
            neural[image_id][digit] = float(atom["value"])
        elif predicate == "IMAGESUM":
            image_1, image_2, sum_value = args
            image_sum[(image_1, image_2)][sum_value] = float(atom["value"])
    return neural, image_sum


def entity_lookup(path):
    rows = load_psl(path, dtype=float)
    return {int(row[0]): (np.asarray(row[1:-1], dtype=np.float32), int(row[-1])) for row in rows}


def true_sum_lookup(path):
    truth = {}
    for row in load_psl(path, dtype=int):
        image_1, image_2, sum_value, is_true = row
        if is_true == 1:
            truth[(image_1, image_2)] = sum_value
    return truth


def argmax_label(values):
    return int(np.argmax(np.asarray(values, dtype=float)))


def round_probs(values, digits=4):
    return [round(float(value), digits) for value in values]


neural_atoms, image_sum_atoms = collect_atoms(payload)
entities = entity_lookup(data_dir / "entity-data-map.txt")
true_sums = true_sum_lookup(data_dir / "image-sum-truth-test.txt")
len(neural_atoms), len(image_sum_atoms)

## Multiple Inference Examples

The examples below show several addition pairs. Each row contains the two MNIST images and the PSL-fused sum soft vector.

In [ ]:
def digit_report(image_id):
    image_features, true_digit = entities[image_id]
    cnn_probs = model.predict(image_features.reshape(1, -1))[0]
    psl_probs = [neural_atoms[image_id].get(digit, 0.0) for digit in range(config.class_size)]
    return {
        "image_id": image_id,
        "features": image_features,
        "true_digit": true_digit,
        "cnn_probs": cnn_probs.tolist(),
        "cnn_pred": argmax_label(cnn_probs),
        "psl_probs": psl_probs,
        "psl_pred": argmax_label(psl_probs),
    }



def pair_report(pair):
    image_1, image_2 = pair
    d1 = digit_report(image_1)
    d2 = digit_report(image_2)
    sum_probs = [image_sum_atoms[pair].get(sum_value, 0.0) for sum_value in range(config.max_sum + 1)]
    return {
        "pair": pair,
        "digit_1": d1,
        "digit_2": d2,
        "sum_probs": sum_probs,
        "sum_pred": argmax_label(sum_probs),
        "true_sum": true_sums[pair],
    }


num_examples = 6
example_pairs = sorted(image_sum_atoms.keys())[:num_examples]
examples = [pair_report(pair) for pair in example_pairs]

for index, example in enumerate(examples, start=1):
    d1 = example["digit_1"]
    d2 = example["digit_2"]
    print(f"\nExample {index}: image {d1['image_id']} + image {d2['image_id']}")
    print(f"真实: {d1['true_digit']} + {d2['true_digit']} = {example['true_sum']}")
    print(f"Transformer预测: {d1['cnn_pred']} + {d2['cnn_pred']} = {d1['cnn_pred'] + d2['cnn_pred']}")
    print(f"PSL digit预测: {d1['psl_pred']} + {d2['psl_pred']} = {d1['psl_pred'] + d2['psl_pred']}")
    print(f"PSL ImageSum预测: {example['sum_pred']}")
    print(f"PSL融合后加法软标签: {round_probs(example['sum_probs'])}")

In [ ]:
sum_values = np.arange(config.max_sum + 1)
fig = plt.figure(figsize=(13, 2.6 * len(examples)), constrained_layout=True)
subfigs = fig.subfigures(len(examples), 1)
if len(examples) == 1:
    subfigs = [subfigs]

for index, (subfig, example) in enumerate(zip(subfigs, examples), start=1):
    d1 = example["digit_1"]
    d2 = example["digit_2"]
    axes = subfig.subplots(1, 3, gridspec_kw={"width_ratios": [1, 1, 3.2]})

    for ax, digit_row, title in zip(axes[:2], [d1, d2], ["Digit 1", "Digit 2"]):
        ax.imshow(digit_row["features"].reshape(28, 28), cmap="gray")
        ax.set_title(f"{title}\ntrue={digit_row['true_digit']} Transformer={digit_row['cnn_pred']} PSL={digit_row['psl_pred']}", fontsize=9)
        ax.axis("off")

    axes[2].bar(sum_values, example["sum_probs"], color="tab:green")
    axes[2].axvline(example["true_sum"], color="black", linestyle="--", linewidth=1, label="true")
    axes[2].axvline(example["sum_pred"], color="tab:red", linestyle=":", linewidth=1.5, label="pred")
    axes[2].set_xticks(sum_values)
    axes[2].set_ylim(0, max(example["sum_probs"]) * 1.2 if max(example["sum_probs"]) > 0 else 1)
    axes[2].set_xlabel("sum")
    axes[2].set_ylabel("PSL soft truth")
    axes[2].set_title(
        f"Example {index}: true {d1['true_digit']} + {d2['true_digit']} = {example['true_sum']} | "
        f"ImageSum pred={example['sum_pred']}",
        fontsize=10,
    )
    axes[2].legend(loc="upper right", fontsize=8)

plt.show()

## Digit Soft-Label Details

For each example, compare the raw Transformer digit probability vector against the `NeuralClassifier` atoms returned by the runtime.

In [ ]:
digits = np.arange(config.class_size)
width = 0.38
fig, axes = plt.subplots(len(examples), 2, figsize=(12, 2.4 * len(examples)), sharey=True)
if len(examples) == 1:
    axes = np.asarray([axes])

for row_index, example in enumerate(examples):
    for col_index, digit_row in enumerate([example["digit_1"], example["digit_2"]]):
        ax = axes[row_index, col_index]
        ax.bar(digits - width / 2, digit_row["cnn_probs"], width=width, label="Transformer")
        ax.bar(digits + width / 2, digit_row["psl_probs"], width=width, label="PSL")
        ax.axvline(digit_row["true_digit"], color="black", linestyle="--", linewidth=1)
        ax.set_xticks(digits)
        ax.set_ylim(0, 1.0)
        ax.set_title(
            f"Ex {row_index + 1} digit {col_index + 1}: true={digit_row['true_digit']}, "
            f"Transformer={digit_row['cnn_pred']}, PSL={digit_row['psl_pred']}",
            fontsize=9,
        )
        if row_index == len(examples) - 1:
            ax.set_xlabel("digit")
        if col_index == 0:
            ax.set_ylabel("soft value")
        if row_index == 0 and col_index == 1:
            ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()